# 01 · Data overview: tables, keys, coverage, cities and regions

**Goal:** understand the *structure* of the ProtoStar data before looking at values.

| Table | Grain | Source file |
|---|---|---|
| `annual_indices` | one row per city × year | `data/raw/protostar/annual_indices.parquet` |
| `city_info` | one row per **station thread** (a city's record is stitched from several stations) | `city_info.parquet` |
| `daily` | one row per city × day | `daily.parquet` (~10M rows) |

Regions are **approximate**, derived from lat/lon in `proto_star.data.protostar.assign_region`
(the raw data has no state column). Everything here reads raw data and modifies nothing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from proto_star.climate import MAX_MISSING_DAYS, METRICS, annual_to_long
from proto_star.data.protostar import (
    REGIONS,
    city_table,
    load_annual,
    load_city_info,
    load_daily,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
REGION_PALETTE = dict(zip(REGIONS, sns.color_palette("tab10", len(REGIONS)), strict=True))

In [ ]:
annual = load_annual()
city_info = load_city_info()
daily_keys = load_daily(columns=[])  # only ID + Date: cheap way to inspect coverage
cities = city_table(annual, city_info)

for name, table in {"annual": annual, "city_info": city_info, "daily": daily_keys}.items():
    print(f"{name:10s} {table.shape[0]:>11,d} rows x {table.shape[1]} cols")

## 1. Schemas

In [ ]:
annual.dtypes.to_frame("dtype").T

In [ ]:
annual.head()

In [ ]:
city_info.head(8)

## 2. Keys and joins

Checks: is `(ID, Year)` unique? Do the three tables cover the same city IDs?
Are name and coordinates constant per city?

In [ ]:
ids = {
    "annual": set(annual["ID"]),
    "city_info": set(city_info["ID"]),
    "daily": set(daily_keys["ID"].astype(str)),
}
key_checks = {
    "duplicate (ID, Year) rows in annual": int(annual.duplicated(["ID", "Year"]).sum()),
    "duplicate (ID, Date) rows in daily": int(daily_keys.duplicated(["ID", "Date"]).sum()),
    "cities in annual": len(ids["annual"]),
    "cities in city_info": len(ids["city_info"]),
    "cities in daily": len(ids["daily"]),
    "IDs not shared by all three tables": len(
        set.union(*ids.values()) - set.intersection(*ids.values())
    ),
    "IDs with >1 Name": int(annual.drop_duplicates(["ID", "Name"])["ID"].duplicated().sum()),
    "Names shared by >1 ID": int(annual.drop_duplicates(["ID", "Name"])["Name"].duplicated().sum()),
    "IDs with >1 (Lat, Lon)": int(
        annual.drop_duplicates(["ID", "Lat", "Lon"])["ID"].duplicated().sum()
    ),
}
pd.Series(key_checks, name="value").to_frame()

**Note:** `Lat`/`Lon` are a single coordinate per city (the *current* station), even though
older years were measured at other stations (see section 5). Coordinates should not be read
as the location of historical measurements.

## 3. Cities and regions

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(
    data=cities,
    x="Lon",
    y="Lat",
    hue="region",
    size="n_years",
    sizes=(20, 160),
    palette=REGION_PALETTE,
    ax=ax,
)
for _, row in cities.sample(25, random_state=0).iterrows():
    ax.annotate(row["Name"], (row["Lon"], row["Lat"]), fontsize=7, alpha=0.7)
ax.set_title("210 ProtoStar cities (size = years of record, colour = approximate region)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

In [ ]:
cities.groupby("region", observed=True).agg(
    n_cities=("ID", "size"),
    median_first_year=("first_year", "median"),
    median_years=("n_years", "median"),
    median_threads=("n_station_threads", "median"),
    lat_range=("Lat", lambda s: f"{s.min():.1f} to {s.max():.1f}"),
)

## 4. Temporal coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(cities["first_year"], bins=25, ax=axes[0])
axes[0].set_title("First year of record per city")

per_year = annual.groupby("Year")["ID"].nunique()
per_year.plot(ax=axes[1])
axes[1].set_title("Cities with an annual row, by year")
axes[1].set_ylabel("cities")
plt.tight_layout()
print("last year per city:", cities["last_year"].unique())

In [ ]:
# Is the daily table a complete calendar (missing days as NaN) or does it drop rows?
span = daily_keys.groupby("ID", observed=True)["Date"].agg(["min", "max", "size"])
span["expected_days"] = (span["max"] - span["min"]).dt.days + 1
span["gap_days"] = span["expected_days"] - span["size"]
print("cities with absent calendar days:", int((span["gap_days"] != 0).sum()))
span.describe()

## 5. Station threads (record stitching)

Each city's record is a chain of stations (`Stn.stDate` → `Stn.edDate`). A station move can
cause a step change in the series, which a model could mistake for climate signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(x=cities["n_station_threads"], ax=axes[0], color="tab:blue")
axes[0].set_title("Station threads per city")

switches = city_info.sort_values(["ID", "Stn.stDate"]).groupby("ID").tail(-1)
sns.histplot(switches["Stn.stDate"].dt.year, bins=np.arange(1860, 2025, 5), ax=axes[1])
axes[1].set_title("Year of each station switch")
plt.tight_layout()

last_switch = switches.groupby("ID")["Stn.stDate"].max().dt.year
print(
    f"cities whose current station started in 1989-2018 (last 30y): {(last_switch >= 1989).sum()}"
)

In [ ]:
# Timeline for the most-stitched cities
top = cities.nlargest(12, "n_station_threads")["ID"]
threads = city_info[city_info["ID"].isin(top)].sort_values(["Name", "Stn.stDate"])
fig, ax = plt.subplots(figsize=(12, 5))
for i, (_name, group) in enumerate(threads.groupby("Name")):
    for j, (_, row) in enumerate(group.iterrows()):
        ax.barh(
            i,
            row["Stn.edDate"] - row["Stn.stDate"],
            left=row["Stn.stDate"],
            color=sns.color_palette("tab20")[j % 20],
            edgecolor="white",
        )
    ax.text(pd.Timestamp("2020-06-01"), i, f"{len(group)}", va="center", fontsize=8)
ax.set_yticks(range(threads["Name"].nunique()), sorted(threads["Name"].unique()))
ax.set_title("Station threads over time, 12 most-stitched cities (count at right)")
plt.tight_layout()

In [ ]:
# Do thread date ranges tile the record without gaps/overlaps?
ordered = city_info.sort_values(["ID", "Stn.stDate"]).copy()
ordered["gap_days"] = (
    ordered["Stn.stDate"] - ordered.groupby("ID")["Stn.edDate"].shift(1)
).dt.days - 1
gaps = ordered.dropna(subset=["gap_days"])
print("thread boundaries:", len(gaps))
print("  with gaps  (>0 days):", int((gaps["gap_days"] > 0).sum()))
print("  overlapping (<0 days):", int((gaps["gap_days"] < 0).sum()))
gaps.loc[gaps["gap_days"] != 0, ["Name", "Stn.Name", "Stn.stDate", "gap_days"]].head(10)

## Findings: structure

- **The keys are clean:** 210 cities appear in all three tables, there are no duplicate
  `(ID, Year)` or `(ID, Date)` rows, and each ID has exactly one name and one coordinate pair.
- **The panel is ragged at the start but aligned at the end:** records begin between 1851 and 1899
  and all end in 2018. That gives 120 to 168 years per city.
- **Approximate regions are fairly balanced** except Southwest, which has only 14 cities.
  Regions come from lat/lon cut-offs, because the data has no state field. If regions turn out
  to matter, consider joining a proper state or climate-division lookup and documenting it in
  docs/data.md.
- **Records are stitched from several stations:** 180 of 210 cities have at least 2 station
  threads, and one city has 10. **30 cities switched station within the 1989-2018 baseline window.**
- **Thread metadata has inconsistencies:** of 251 boundaries, 7 have gaps (for example Sioux City
  has a 36,524-day gap) and 1 overlaps (Abilene, by 366 days). These records need cleaning before
  station-switch dates are used as features.
- **Coordinates belong to the current station only,** not to the historical ones.